### Import Dependencies

In [ ]:
from openai import OpenAI
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
openai_client = OpenAI()
typesafe_client = TypeSafeClient()

## Model / Intent Routing

In [ ]:
query = "I am looking for a charger comaptible with iPhone 15"

#### OpenAI Decisions API

In [ ]:
response_choice_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=query,
    questions=[
        {
            "type": "choice",
            "name": "category",
            "instructions": "Classify the usr intent.",
            "choices": [
                {"value": "product_search", "description": "Search for a product"},
                {"value": "order_status", "description": "Check the status of an order"},
                {"value": "returns", "description": "Return a product"},
                {"value": "general", "description": "Anything else"},
            ],
        }
    ]
)

#### TypeSafeAI SDK

In [ ]:
response_choice_ts = typesafe_client.system_one(
    state=query,
    questions={
        "category": Choice(
            instructions="Classify the usr intent.",
            criteria={
                "product_search": "Search for a product",
                "order_status": "Check the status of an order",
                "returns": "Return a product",
                "general": "Anything else"
            }
        )
    }
)

In [ ]:
CALIBRATED_THRESHOLD = 0.75
route = response_choice_ts.answers["category"].choice if response_choice_ts.answers["category"].confidence >= CALIBRATED_THRESHOLD else "fallback"

In [ ]:
route

## Guardrails

In [ ]:
USER_MESSAGE = "I am John Smith, security number 1234567890 and I want to return my order"

HAZARDS = {
    "injection": {
        "description": "The input tries to override the system instructions", 
        "threshold": 0.40
    },
    "pii": {
        "description": "The input contains personal identifying information", 
        "threshold": 0.60
    },
    "off_policy": {
        "description": "The input asks for medical, legal or financial advice", 
        "threshold": 0.70
    },
    "abuse": {
        "description": "The input is abusive toward the assistant or staff", 
        "threshold": 0.90
    },
}

#### OpenAI Decisions API

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=USER_MESSAGE,
    questions=[
        {
            "type": "predicate",
            "name": name,
            "instructions": hazard["description"],
        }
        for name, hazard in HAZARDS.items()
    ]
)

In [ ]:
block = any(hazard.probability >= HAZARDS[hazard.name]["threshold"] for hazard in response_predicate_oai.answers)

In [ ]:
block

#### TypeSafeAI SDK

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=USER_MESSAGE,
    questions={
        name: Noul(
            instructions=hazard["description"],
        )
        for name, hazard in HAZARDS.items()
    }
)

In [ ]:
block = any(hazard.noul >= HAZARDS[name]["threshold"] for name, hazard in response_noul_ts.answers.items())

In [ ]:
block

## Retrieval Sufficiency

In [ ]:
USER_MESSAGE = "I am looking for a charger comaptible with iPhone 15"

RETRIEVED_PASSAGES = [
    "The headphones have no noise cancellation capability",
    "The charger A is compatible with Samsung Galaxy S23 only",
    "The headphones have a 3.5mm audio jack",
    "The charger B is compatible with Apple iPhone 15"
]

#### OpenAI Decisions API

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=f"question: {USER_MESSAGE}\nretrieved passages: {RETRIEVED_PASSAGES}",
    questions=[
        {
            "type": "predicate",
            "name": "sufficient",
            "instructions": "Retrieved passages are enought to answer the question.",
        }
    ]
)

In [ ]:
CALIBRATED_THRESHOLD = 0.9
retrieved_data_sufficient = response_predicate_oai.answers[0].probability >= CALIBRATED_THRESHOLD

In [ ]:
retrieved_data_sufficient

#### TypeSafeAI SDK

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=f"question: {USER_MESSAGE}\nretrieved passages: {RETRIEVED_PASSAGES}",
    questions={
        "sufficient": Noul(
            instructions="Retrieved passages are enought to answer the question.",
        )
    }
)

In [ ]:
CALIBRATED_THRESHOLD = 0.9
retrieved_data_sufficient = response_noul_ts.answers["sufficient"].noul >= CALIBRATED_THRESHOLD

In [ ]:
retrieved_data_sufficient

## Passage Filtering and Reranking

In [ ]:
USER_MESSAGE = "I am looking for a charger comaptible with iPhone 15"

RETRIEVED_PASSAGES = [
    "The headphones have no noise cancellation capability",
    "The charger A is compatible with Samsung Galaxy S23 only",
    "The headphones have a 3.5mm audio jack",
    "The charger B is compatible with Apple iPhone 15"
]

#### OpenAI Decisions API

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=f"question: {USER_MESSAGE}",
    questions=[
        {
            "type": "predicate",
            "name": f"{index}",
            "instructions": f"The passage: {passage} is relevant to the question.",
        }
        for index, passage in enumerate(RETRIEVED_PASSAGES)
    ]
)

In [ ]:
CALIBRATED_THRESHOLD = 0.5
relevant_passages = [RETRIEVED_PASSAGES[int(passage.name)] for passage in response_predicate_oai.answers if passage.probability >= CALIBRATED_THRESHOLD]

In [ ]:
relevant_passages

#### TypeSafeAI SDK

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=f"question: {USER_MESSAGE}",
    questions={
        f"{index}": Noul(
            instructions=f"The passage: {passage} is relevant to the question.",
        )
        for index, passage in enumerate(RETRIEVED_PASSAGES)
    }
)

In [ ]:
CALIBRATED_THRESHOLD = 0.5
relevant_passages = [RETRIEVED_PASSAGES[int(id)] for id, passage in response_noul_ts.answers.items() if passage.noul >= CALIBRATED_THRESHOLD]

In [ ]:
relevant_passages

## Citation Checking

In [ ]:
FULL_DRAFT = "The Anker 737 charges the MacBook Pro 16 at full speed and ships with a USB-C cable."
CLAIMS = [
    "The Anker 737 charges the MacBook Pro 16 at full speed.",
    "The Anker 737 ships with a USB-C cable.",
]
PASSAGES = [
    "Product description: The Anker 737 is a 140W USB-C charger. Capable of charging the MacBook Pro 16 at full speed.",
    "User review: I had to buy the cable separately :(",
]

#### OpenAI Decisions API

In [ ]:
response_choice_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=f"Available passages: {PASSAGES}",
    questions=[
        {
            "type": "choice",
            "name": f"{index}",
            "instructions": f"The claim: '{claim}' is supported by the passages.",
            "choices": [
                {"value": "supports_claim", "description": "The claim is supported by the passages"},
                {"value": "contradicts_claim", "description": "The claim contradicts the passages"},
                {"value": "does_not_address_claim", "description": "The claim is not addressed by the passages"},
            ],
        }
        for index, claim in enumerate(CLAIMS)
    ]
)

In [ ]:
regenerate = False
claims_to_strip = []
CALIBRATED_THRESHOLD = 0.5
for claim in response_choice_oai.answers:
    if claim.confidence >= CALIBRATED_THRESHOLD:
        if claim.choice == "contradicts_claim":
            regenerate = True
        elif claim.choice == "does_not_address_claim":
            claims_to_strip.append(CLAIMS[int(claim.name)])

In [ ]:
print(f"Regenerate: {regenerate}")
print(f"Claims to strip: {claims_to_strip}")

#### TypeSafeAI SDK

In [ ]:
response_choice_ts = typesafe_client.system_one(
    state=f"Available passages: {PASSAGES}",
    questions={
        f"{index}": Choice(
            instructions=f"The claim: '{claim}' is supported by the passages.",
            criteria={
                "supports_claim": "The claim is supported by the passages",
                "contradicts_claim": "The claim contradicts the passages",
                "does_not_address_claim": "The claim is not addressed by the passages",
            }
        )
        for index, claim in enumerate(CLAIMS)
    }
)

In [ ]:
regenerate = False
claims_to_strip = []
CALIBRATED_THRESHOLD = 0.5
for name, claim in response_choice_ts.answers.items():
    if claim.confidence >= CALIBRATED_THRESHOLD:
        if claim.choice == "contradicts_claim":
            regenerate = True
        elif claim.choice == "does_not_address_claim":
            claims_to_strip.append(CLAIMS[int(name)])

In [ ]:
print(f"Regenerate: {regenerate}")
print(f"Claims to strip: {claims_to_strip}")

## Tool-Call Gating

In [ ]:
PROPOSED_TOOL_CALL = {
    "name": "search_products",
    "arguments": {
        "query": "charger compatible with iPhone 15"
    }
}
CONVERSATION_HISTORY = [
    {
        "role": "user",
        "content": "I am looking for a charger comaptible with iPhone 15"
    }
]

#### OpenAI Decisions API

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=f"Conversation history: {CONVERSATION_HISTORY}\nProposed tool call: {PROPOSED_TOOL_CALL}",
    questions=[
        {
            "type": "predicate",
            "name": "irreversible",
            "instructions": "The action proposed by the tool call cannot be undone.",
        },
        {
            "type": "predicate",
            "name": "intended",
            "instructions": "The action proposed by the tool call is intended according to the conversation history.",
        }
    ]
)

In [ ]:
INTENDED_THRESHOLD = 0.95
IRREVERSIBLE_THRESHOLD = 0.80
intended = False
irreversible = False
for answer in response_predicate_oai.answers:
    if answer.name == "intended" and answer.probability >= INTENDED_THRESHOLD:
        intended = True
    if answer.name == "irreversible" and answer.probability >= IRREVERSIBLE_THRESHOLD:
        irreversible = True
allow_tool_call = intended and not irreversible
escalate_to_human = intended and irreversible
block_tool_call = not intended

In [ ]:
print(f"Allow tool call: {allow_tool_call}")
print(f"Escalate to human: {escalate_to_human}")
print(f"Block tool call: {block_tool_call}")

#### TypeSafeAI SDK

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=f"Conversation history: {CONVERSATION_HISTORY}\nProposed tool call: {PROPOSED_TOOL_CALL}",
    questions={
        "irreversible": Noul(
            instructions=f"The action proposed by the tool call cannot be undone.",
        ),
        "intended": Noul(
            instructions=f"The action proposed by the tool call is intended according to the conversation history.",
        )      
    }
)

In [ ]:
INTENDED_THRESHOLD = 0.95
IRREVERSIBLE_THRESHOLD = 0.80
intended = False
irreversible = False
for name, answer in response_noul_ts.answers.items():
    if name == "intended" and answer.noul >= INTENDED_THRESHOLD:
        intended = True
    if name == "irreversible" and answer.noul >= IRREVERSIBLE_THRESHOLD:
        irreversible = True
allow_tool_call = intended and not irreversible
escalate_to_human = intended and irreversible
block_tool_call = not intended

In [ ]:
print(f"Allow tool call: {allow_tool_call}")
print(f"Escalate to human: {escalate_to_human}")
print(f"Block tool call: {block_tool_call}")

## Loop Termination

In [ ]:
TERMINATION_CRITERIA = {
    "options": "The assistant has returned at least three in-stock options.",
    "budget": "If stated by the user, every option named is under the budget."
}

USER_REQUEST = "I am looking for a charger comaptible with iPhone 15, under $25."
AGENT_RESPONSE = """
Here are the options I found:
- Anker 737: $20
- Anker 738: $22
"""

#### OpenAI Decisions API

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=f"User request: {USER_REQUEST}\nAgent response: {AGENT_RESPONSE}",
    questions=[
        {
            "type": "predicate",
            "name": name,
            "instructions": f"Does the agent response meet the criterion: '{criterion}'?",
        }
        for name, criterion in TERMINATION_CRITERIA.items()
    ]
)

In [ ]:
CALIBRATED_THRESHOLD = 0.5
failed_criteria = [criterion.name for criterion in response_predicate_oai.answers if criterion.probability < CALIBRATED_THRESHOLD]

terminate_loop = False
additional_context = ""
if len(failed_criteria) > 0:
    additional_context = f"The following criteria failed: {', '.join([TERMINATION_CRITERIA[criterion] for criterion in failed_criteria])}"
else:
    terminate_loop = True

print(f"Terminate loop: {terminate_loop}")
print(f"Additional context: {additional_context}")

#### TypeSafeAI SDK

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=f"User request: {USER_REQUEST}\nAgent response: {AGENT_RESPONSE}",
    questions={
        name: Noul(
            instructions=f"Does the agent response meet the criterion: '{criterion}'?",
        )
        for name, criterion in TERMINATION_CRITERIA.items()
    }
)

In [ ]:
CALIBRATED_THRESHOLD = 0.5
failed_criteria = [name for name, criterion in response_noul_ts.answers.items() if criterion.noul >= CALIBRATED_THRESHOLD]

terminate_loop = False
additional_context = ""
if len(failed_criteria) > 0:
    additional_context = f"The following criteria failed: {', '.join([TERMINATION_CRITERIA[criterion] for criterion in failed_criteria])}"
else:
    terminate_loop = True

print(f"Terminate loop: {terminate_loop}")
print(f"Additional context: {additional_context}")

## Context Compaction

In [ ]:
PAST_OBJECTIVES = [
    "I am looking for a charger comaptible with iPhone 15"
]
RETRIEVED_PASSAGES = [
    "The headphones have noise cancellation capability",
    "The charger A is compatible with Samsung Galaxy S23 only",
    "The headphones have a 3.5mm audio jack",
    "The charger B is compatible with Apple iPhone 15"
]
CURRENT_OBJECTIVE = "Add the item with id B to the cart"

#### OpenAI Decisions API

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=f"Current objective: {CURRENT_OBJECTIVE}\nPast objectives: {PAST_OBJECTIVES}",
    questions=[
        {
            "type": "predicate",
            "name": f"{index}",
            "instructions": f"The passage: '{passage}' might be relevant to the current and future objectives.",
        }
        for index, passage in enumerate(RETRIEVED_PASSAGES)
    ]
)

In [ ]:
CALIBRATED_THRESHOLD = 0.4
passages_to_discard = [RETRIEVED_PASSAGES[int(passage.name)] for passage in response_predicate_oai.answers if passage.probability < CALIBRATED_THRESHOLD]
passages_to_discard

#### TypeSafeAI SDK

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=f"Current objective: {CURRENT_OBJECTIVE}\nPast objectives: {PAST_OBJECTIVES}",
    questions={
        f"{index}": Noul(
            instructions=f"The passage: '{passage}' might be relevant to the current and future objectives.",
        )
        for index, passage in enumerate(RETRIEVED_PASSAGES)
    }
)

In [ ]:
CALIBRATED_THRESHOLD = 0.4
passages_to_discard = [RETRIEVED_PASSAGES[int(id)] for id, passage in response_noul_ts.answers.items() if passage.noul < CALIBRATED_THRESHOLD]
passages_to_discard

## Evals

In [ ]:
USER_REQUEST = "I am looking for a charger comaptible with the MacBook Pro 16."
FULL_DRAFT = "The Anker 737 charges the MacBook Pro 16 at full speed and ships with a USB-C cable."
PASSAGES = [
    "Product description: The Anker 737 is a 140W USB-C charger. Capable of charging the MacBook Pro 16 at full speed.",
    "User review: I had to buy the cable separately :(",
]
RUBRIC = {
    "grounded": "Every factual claim in the answer is supported by the available passages.",
    "complete": "The answer addresses every part of the user's question."
}

#### OpenAI Decisions API

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=f"User request: {USER_REQUEST}\nAnswer: {FULL_DRAFT}\nAvailable passages: {PASSAGES}",
    questions=[
        {
            "type": "predicate",
            "name": rubric_name,
            "instructions": rubric_description,
        }
        for rubric_name, rubric_description in RUBRIC.items()
    ]
)

In [ ]:
CALIBRATED_THRESHOLD = 0.5
failed_rubrics = [rubric.name for rubric in response_predicate_oai.answers if rubric.probability < CALIBRATED_THRESHOLD]

In [ ]:
failed_rubrics

#### TypeSafeAI SDK

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=f"User request: {USER_REQUEST}\nAnswer: {FULL_DRAFT}\nAvailable passages: {PASSAGES}",
    questions={
        rubric_name: Noul(
            instructions=rubric_description,
        )
        for rubric_name, rubric_description in RUBRIC.items()
    }
)

In [ ]:
CALIBRATED_THRESHOLD = 0.5
failed_rubrics = [name for name, rubric in response_noul_ts.answers.items() if rubric.noul < CALIBRATED_THRESHOLD]

In [ ]:
failed_rubrics